# Mesozoic Labs: Triceratops 60 cm, walking-policy training (simulation only)

This notebook trains a PPO walking policy for the 60 cm Triceratops build model in MuJoCo on a **CPU** runtime and keeps every file on Google Drive under `MyDrive/mesozoic-labs/triceratops60/<RUN_ID>`. **Simulation only: nothing here touches a robot.**

Follow the step sheet `hardware/triceratops_build/rl/COLAB_STEPS.md`. In short: fill in cell 1, then Runtime > Run all.

* Nothing runs when the notebook opens. Training starts only when cell 5 runs (Run all runs it), and only after cells 1 to 4 have checked the parameters, the code version, the packages and the run folder.
* Cell 5 runs `rl/train.py` in a subprocess. If the runtime disconnects, open the same copy and run all cells again with the same values: cell 5 finds the checkpoints on Drive and resumes from the newest intact one.
* Runtime type: CPU. A GPU adds nothing to this training.
* Untested on Colab itself: the cells were run locally (outside Colab), where the Colab-only parts (clone, pip, Drive, disconnect) are skipped.

In [ ]:
# ===== 1. PARAMETERS: COLAB_STEPS.md gives every value; change nothing else =====
import json
import os
import re

REPO_REF = ""  # @param {"type":"string"}
RUN_ID = ""  # @param {"type":"string"}
TOTAL_STEPS = 20000000  # @param {"type":"integer"}
N_ENVS = 8  # @param {"type":"integer"}
SEED = 0  # @param {"type":"integer"}
EVAL_EPISODES = 10  # @param {"type":"integer"}
USE_GOOGLE_DRIVE = True  # @param {"type":"boolean"}
AUTO_DISCONNECT = True  # @param {"type":"boolean"}
# REPO_REF: the 40 hex characters of the commit to run.  RUN_ID: letters, digits, "-" and "_", such as trike60-r1;
# a new RUN_ID starts a new run, the same RUN_ID resumes it.  TOTAL_STEPS: the run's total environment steps (a
# resumed run stops at the same total).  N_ENVS: parallel simulations, about one per vCPU; keep it the same for
# every session of a run.

NAMES = ("REPO_REF", "RUN_ID", "TOTAL_STEPS", "N_ENVS", "SEED", "EVAL_EPISODES", "USE_GOOGLE_DRIVE", "AUTO_DISCONNECT")
PARAMS = {name: globals()[name] for name in NAMES}
if os.environ.get("TRIKE60_PARAMS"):  # local runs and tests only: a JSON file whose values replace the ones above
    with open(os.environ["TRIKE60_PARAMS"], encoding="utf-8") as _f:
        PARAMS.update(json.load(_f))
    globals().update({name: PARAMS[name] for name in NAMES})

if not re.fullmatch(r"[0-9a-f]{40}", str(REPO_REF)):
    raise ValueError("REPO_REF must be the 40 hex characters of a commit (COLAB_STEPS.md gives it). Nothing was run.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", str(RUN_ID)):
    raise ValueError("RUN_ID must be letters, digits, '-' and '_' (for example trike60-r1). Nothing was run.")
if int(TOTAL_STEPS) <= 0 or int(N_ENVS) < 1 or int(EVAL_EPISODES) < 1:
    raise ValueError("TOTAL_STEPS, N_ENVS and EVAL_EPISODES must be positive. Nothing was run.")
print("Parameters of this session:")
for _name in NAMES:
    print(f"  {_name} = {PARAMS[_name]!r}")

In [ ]:
# ===== 2. CODE: the repository at exactly REPO_REF =====
import pathlib
import signal
import subprocess
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
REPO_URL = "https://github.com/kuds/mesozoic-labs.git"


def run_step(cmd, cwd=None, env=None):
    """Run cmd with its output shown here line by line; on the stop button, ask it to stop cleanly (it saves)."""
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            cwd=cwd, env=env)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        return proc.wait()
    except KeyboardInterrupt:
        print("\nStop requested: the step is asked to stop (train.py first saves a checkpoint pair)...", flush=True)
        proc.send_signal(signal.SIGINT)
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
        raise


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if IN_COLAB:
    REPO_DIR = pathlib.Path("/content/mesozoic-labs")
    if not (REPO_DIR / ".git").exists():
        if run_step(["git", "clone", "--no-checkout", REPO_URL, REPO_DIR]) != 0:
            raise RuntimeError("git clone failed; read the lines above.")
    if run_step(["git", "fetch", "--quiet", "origin", REPO_REF], cwd=REPO_DIR) != 0:
        raise RuntimeError(f"git could not fetch {REPO_REF}; check REPO_REF against COLAB_STEPS.md.")
    fetched = git("rev-parse", "FETCH_HEAD^{commit}")
    if (REPO_DIR / ".git" / "index").exists() and git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError("The checkout has local edits; use a fresh runtime (Runtime > Disconnect and delete runtime).")
    if run_step(["git", "checkout", "--quiet", "--detach", fetched], cwd=REPO_DIR) != 0:
        raise RuntimeError("git checkout failed; read the lines above.")
    HEAD = git("rev-parse", "HEAD")
    CLEAN = not git("status", "--porcelain", "--untracked-files=no")
    print(f"Repository commit: {HEAD} (matches REPO_REF: {'yes' if HEAD == REPO_REF else 'NO'}; "
          f"checkout clean: {'yes' if CLEAN else 'NO'})")
    if HEAD != REPO_REF or not CLEAN:
        raise RuntimeError("The checkout is not exactly REPO_REF. Stop and report this line.")
else:
    REPO_DIR = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                    if (p / "hardware" / "triceratops_build" / "rl" / "train.py").exists())
    HEAD = git("rev-parse", "HEAD")
    print(f"Not on Colab: using the checkout at {REPO_DIR} (commit {HEAD}); REPO_REF only selects the Colab checkout.")
RL_DIR = REPO_DIR / "hardware" / "triceratops_build" / "rl"
# every step below is a subprocess with this environment: no OpenGL, one thread per process
STEP_ENV = dict(os.environ, MUJOCO_GL="disable", PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")

In [ ]:
# ===== 3. PACKAGES: the pins of rl/requirements-colab.txt, torch as the CPU build =====
REQ = RL_DIR / "requirements-colab.txt"
TORCH_CPU_INDEX = "https://download.pytorch.org/whl/cpu"
PINS = {}
for _line in REQ.read_text().splitlines():
    _line = _line.split("#", 1)[0].strip()
    if _line:
        _name, _version = _line.split("==")
        PINS[_name.strip()] = _version.strip()


def installed_versions():
    """The installed versions, read in a fresh interpreter (this kernel may still hold the old ones)."""
    code = ("import importlib.metadata as m, json, sys\nout = {}\nfor n in sys.argv[1:]:\n    try:\n"
            "        out[n] = m.version(n)\n    except m.PackageNotFoundError:\n        out[n] = None\nprint(json.dumps(out))")
    return json.loads(subprocess.check_output([sys.executable, "-c", code, *PINS], text=True))


def differences(vers):
    return [f"{n}: installed {vers.get(n)}, pinned {v}" for n, v in PINS.items()
            if (vers.get(n) or "").split("+", 1)[0] != v]


_before = installed_versions()
if IN_COLAB:
    if _before.get("torch") != f"{PINS['torch']}+cpu":  # pip would take an installed 2.x.y+cu... as ==2.x.y
        if run_step([sys.executable, "-m", "pip", "install", "-q", f"torch=={PINS['torch']}+cpu",
                     "--index-url", TORCH_CPU_INDEX]) != 0:
            raise RuntimeError("torch (CPU build) did not install; send the lines above.")
    if differences(installed_versions()):
        if run_step([sys.executable, "-m", "pip", "install", "-q", "-r", REQ]) != 0:
            raise RuntimeError("A pinned package did not install; send the lines above.")
VERSIONS = installed_versions()
_wrong = differences(VERSIONS)
if IN_COLAB and not (VERSIONS.get("torch") or "").endswith("+cpu"):
    _wrong.append(f"torch: installed {VERSIONS.get('torch')}, not the CPU build")
print("Packages: " + ", ".join(f"{n} {VERSIONS.get(n)}" for n in PINS))
if _wrong and IN_COLAB:
    raise RuntimeError("Pins do not hold: " + "; ".join(_wrong))
print("All pins hold: " + ("yes" if not _wrong else "NO (not on Colab, so not enforced): " + "; ".join(_wrong)))

In [ ]:
# ===== 4. STORAGE: Google Drive and the run folder =====
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive/mesozoic-labs/triceratops60")
elif IN_COLAB:
    BASE = pathlib.Path("/content/triceratops60")
    print(f"Warning: USE_GOOGLE_DRIVE is off. The files go to {BASE} and are lost with the runtime.")
else:
    BASE = pathlib.Path(PARAMS.get("LOCAL_RUNS_DIR") or pathlib.Path.home() / "mesozoic-labs-runs" / "triceratops60")
RUN_DIR = BASE / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
_record = RUN_DIR / "run_record.json"
if _record.exists():
    _first = json.loads(_record.read_text())["sessions"][0]["git"]["commit"]
    if _first != HEAD and IN_COLAB:
        raise RuntimeError(f"Run {RUN_ID} was started from commit {_first}, not {HEAD}. A run keeps its commit: "
                           "use the run's own REPO_REF, or a new RUN_ID. Nothing was changed.")
    if _first != HEAD:
        print(f"Note: run {RUN_ID} was started from commit {_first}; this checkout is at {HEAD} (not enforced "
              "outside Colab).")
_ckpts = sorted((RUN_DIR / "checkpoints").glob("trike60_*_steps.zip"))
print(f"Run folder: {RUN_DIR} ({'resuming: ' + str(len(_ckpts)) + ' checkpoint(s) found' if _ckpts else 'new run'})")

In [ ]:
# ===== 5. TRAINING: rl/train.py in a subprocess; it resumes by itself when the run folder holds checkpoints =====
_ckpts = sorted((RUN_DIR / "checkpoints").glob("trike60_*_steps.zip"))
TRAIN_CMD = [sys.executable, "-u", RL_DIR / "train.py", "--out", RUN_DIR, "--total-steps", int(TOTAL_STEPS),
             "--n-envs", int(N_ENVS), "--seed", int(SEED)] + (["--resume"] if _ckpts else [])
print("Running: train.py " + " ".join(str(c) for c in TRAIN_CMD[3:]), flush=True)
_code = run_step(TRAIN_CMD, cwd=REPO_DIR, env=STEP_ENV)
if _code != 0:
    raise RuntimeError(f"train.py stopped with exit code {_code}; the reason is in the lines above. "
                       "What it saved is in the run folder.")

In [ ]:
# ===== 6. EVALUATION: EVAL_EPISODES episodes, randomisation on, servo delay 10 ms (and 0 ms) =====
EVAL_CMD = [sys.executable, "-u", RL_DIR / "evaluate.py", "--run-dir", RUN_DIR, "--episodes", int(EVAL_EPISODES),
            "--delays", 10, 0]
_code = run_step(EVAL_CMD, cwd=REPO_DIR, env=STEP_ENV)
if _code != 0:
    raise RuntimeError(f"evaluate.py stopped with exit code {_code}; the reason is in the lines above.")
print(f"Results: {RUN_DIR / 'eval'}")

In [ ]:
# ===== 7. FINISH: flush Drive and release the runtime =====
try:
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    from environments.shared.notebook_runtime import disconnect_runtime
except Exception as _exc:  # this kernel's own packages may be stale after cell 3: release the runtime directly
    print(f"(the shared helper did not import here: {_exc!r}; releasing the runtime directly)")

    def disconnect_runtime(reason, *, in_colab, auto, flush_drive):
        import time

        print(f"\n{reason}")
        if not (in_colab and auto):
            print("Auto-disconnect skipped (not in Colab or AUTO_DISCONNECT is False).")
            return
        if flush_drive:
            from google.colab import drive

            print("Flushing Google Drive writes...")
            drive.flush_and_unmount()
        from google.colab import runtime

        print("Disconnecting runtime in 5 seconds...")
        time.sleep(5)
        runtime.unassign()

disconnect_runtime(f"Run {RUN_ID} is finished; its files are in {RUN_DIR}.", in_colab=IN_COLAB, auto=AUTO_DISCONNECT,
                   flush_drive=IN_COLAB and USE_GOOGLE_DRIVE)